# Capstone Project 2: Titanic Survival — Missing Data, Feature Engineering, and Model Comparison

The second capstone uses the classic Titanic dataset (already present in
`4. ML Algorithms/2. Classification/Classroom/titanic_dataset/`) to focus on a part of
the pipeline the Heart Disease capstone skipped: **real missing data** and the feature
engineering decisions that come with it. Unlike the heart disease data, Titanic arrives
genuinely messy — missing ages, mostly-missing cabin numbers, a couple of missing
embarkation ports — and the choices made in cleaning it change the final model's
performance more than the choice of algorithm does.

### Workflow

1. Load and audit missingness
2. Feature engineering: titles from names, family size, cabin deck
3. Impute what should be imputed, drop what should be dropped
4. Train/test split and three classifiers (logistic regression, KNN, random forest)
5. Compare via cross-validation, then a single honest test-set evaluation
6. Which engineered feature mattered most?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from sklearn.inspection import permutation_importance

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)
SKF = StratifiedKFold(5, shuffle=True, random_state=0)

## 1. Load and audit missingness

In [ ]:
df = pd.read_csv("../4. ML Algorithms/2. Classification/Classroom/titanic_dataset/train.csv")
print(df.shape)
df.head()

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)
pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})[missing > 0]

`Cabin` is missing for 77% of passengers -- too much to impute meaningfully, but the
fact that it is *recorded at all* is itself informative (cabin numbers were mostly kept
for higher-class passengers). `Age` is missing for ~20%, worth imputing carefully rather
than dropping a fifth of the data. `Embarked` is missing for just 2 rows -- safe to fill
with the mode.

## 2. Feature engineering

Three engineered features that are standard on this dataset and each teach a different
lesson:

* **Title** (extracted from the name) is a much better proxy for age/social status than
  the raw name string, and helps impute missing ages more accurately than a single
  overall median would.
* **Family size** (`SibSp + Parch + 1`) captures a real effect the raw counts hide:
  survival was worse for people travelling completely alone *and* for people in very
  large families, and better for small families — a non-monotonic relationship a linear
  model cannot see in the raw counts alone.
* **Has cabin recorded** turns the mostly-missing `Cabin` column from "throw it away"
  into a genuinely useful binary signal.

In [ ]:
df["Title"] = df["Name"].str.extract(r",\s*([^.]*)\.")
print(df["Title"].value_counts())

In [ ]:
# Collapse rare titles into a handful of meaningful buckets
title_map = {
    "Mr": "Mr", "Miss": "Miss", "Mrs": "Mrs", "Master": "Master",
    "Dr": "Officer/Noble", "Rev": "Officer/Noble", "Col": "Officer/Noble",
    "Major": "Officer/Noble", "Capt": "Officer/Noble",
    "Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs",
    "Don": "Officer/Noble", "Lady": "Officer/Noble", "Sir": "Officer/Noble",
    "the Countess": "Officer/Noble", "Jonkheer": "Officer/Noble",
}
df["Title"] = df["Title"].map(title_map).fillna("Officer/Noble")
print(df["Title"].value_counts())

df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["HasCabin"] = df["Cabin"].notna().astype(int)
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

# Impute Age using the median WITHIN each title group -- a "Master" (young boy) and a
# "Mr" have very different typical ages, so grouping beats a single overall median.
df["Age"] = df.groupby("Title")["Age"].transform(lambda s: s.fillna(s.median()))
print(f"\nRemaining missing ages: {df['Age'].isnull().sum()}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
sns.barplot(data=df, x="Title", y="Survived", ax=axes[0], errorbar=None)
axes[0].set_title("Survival rate by title"); axes[0].tick_params(axis="x", rotation=30)

fam_survival = df.groupby("FamilySize")["Survived"].mean()
axes[1].plot(fam_survival.index, fam_survival.values, marker="o", color="steelblue")
axes[1].set_xlabel("family size"); axes[1].set_ylabel("survival rate")
axes[1].set_title("Survival vs family size (non-monotonic)")

sns.barplot(data=df, x="HasCabin", y="Survived", ax=axes[2], errorbar=None)
axes[2].set_title("Survival by whether a cabin was recorded")
plt.tight_layout()
plt.show()

The family-size plot is the interesting one: survival rises from "travelling alone" up
to a family of 4, then falls off a cliff for families of 5+ (likely third-class families
who couldn't all reach a lifeboat together). A raw `SibSp`/`Parch` linear coefficient
cannot represent this U-shape; a tree-based model can split on it directly, and even a
linear model benefits from `FamilySize` and `FamilySize**2` as engineered inputs.

## 3. Final feature set

In [ ]:
feature_cols = ["Pclass", "Sex", "Age", "Fare", "FamilySize", "HasCabin", "Embarked", "Title"]
X = df[feature_cols]
y = df["Survived"]

numeric_features = ["Age", "Fare", "FamilySize"]
categorical_features = ["Pclass", "Sex", "HasCabin", "Embarked", "Title"]

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), numeric_features),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features),
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,
                                                     stratify=y)
print(f"train: {len(y_train)}, test: {len(y_test)}, survival rate: {y.mean():.3f}")

## 4. Three models, compared by cross-validation

In [ ]:
models = {
    "baseline": DummyClassifier(strategy="most_frequent"),
    "logistic regression": Pipeline([("prep", preprocess), ("model", LogisticRegression(max_iter=2000))]),
    "k-NN": Pipeline([("prep", preprocess), ("model", KNeighborsClassifier(n_neighbors=15))]),
    "random forest": Pipeline([("prep", preprocess), ("model", RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=0))]),
}

print(f"{'model':<22}{'CV accuracy':>13}{'CV ROC-AUC':>13}")
for name, pipe in models.items():
    acc = cross_val_score(pipe, X_train, y_train, cv=SKF, scoring="accuracy").mean()
    try:
        auc = cross_val_score(pipe, X_train, y_train, cv=SKF, scoring="roc_auc").mean()
    except Exception:
        auc = float("nan")
    print(f"{name:<22}{acc:>13.4f}{auc:>13.4f}")

## 5. Final evaluation on the held-out test set

Only now do we touch the test set — once, for the model that won cross-validation.

In [ ]:
best_name = "random forest"
best_pipe = models[best_name].fit(X_train, y_train)

pred = best_pipe.predict(X_test)
proba = best_pipe.predict_proba(X_test)[:, 1]

print(f"Selected model: {best_name}")
print(f"Test accuracy : {accuracy_score(y_test, pred):.4f}")
print(f"Test ROC-AUC  : {roc_auc_score(y_test, proba):.4f}")
print()
print(classification_report(y_test, pred, target_names=["did not survive", "survived"]))

## 6. Which engineered feature mattered?

In [ ]:
perm = permutation_importance(best_pipe, X_test, y_test, n_repeats=30, random_state=0,
                              scoring="roc_auc")
importance = pd.Series(perm.importances_mean, index=feature_cols).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 4.5))
importance.plot(kind="barh", ax=ax, color="steelblue")
ax.invert_yaxis()
ax.set_title("Permutation importance of the FINAL feature set (after engineering)")
plt.tight_layout()
plt.show()
print(importance.round(4))

In [ ]:
# Compare against the model trained WITHOUT the engineered features, to quantify
# how much the feature engineering step actually bought us.
raw_features = ["Pclass", "Sex", "Age", "Fare", "SibSp", "Parch", "Embarked"]
raw_numeric = ["Age", "Fare", "SibSp", "Parch"]
raw_categorical = ["Pclass", "Sex", "Embarked"]

raw_preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), raw_numeric),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), raw_categorical),
])
raw_pipe = Pipeline([("prep", raw_preprocess),
                     ("model", RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=0))])

X_raw_train = df.loc[X_train.index, raw_features]
raw_cv_auc = cross_val_score(raw_pipe, X_raw_train, y_train, cv=SKF, scoring="roc_auc").mean()
engineered_cv_auc = cross_val_score(best_pipe, X_train, y_train, cv=SKF, scoring="roc_auc").mean()

print(f"CV ROC-AUC without engineered features (Title, FamilySize, HasCabin): {raw_cv_auc:.4f}")
print(f"CV ROC-AUC WITH engineered features                                : {engineered_cv_auc:.4f}")
print(f"\nImprovement from feature engineering: {engineered_cv_auc - raw_cv_auc:+.4f} ROC-AUC")
print("Same algorithm, same hyperparameters -- the entire gap is the three engineered")
print("columns. This is the same lesson as the Linear Regression notebook in the ML")
print("module: feature engineering usually moves the needle more than algorithm choice.")